# 聚會字幕準確率評估（MEET 封存測試，GPU）

用你 Google Drive 裡 MEET 資料夾的測試資料（`01.mp3` ＋ `01.txt`）評估：

- **Whisper large-v3-turbo**（和 Cloudflare 免費 Workers AI 同一個模型）：不加提示／加繁體提示
- 每個結果都比較三種版本：原始 → ＋詞彙表 → ＋同音專名校正（與網站相同的程式碼）
- （可選）若 MEET 裡有 `01.chrome.txt`（網站「下載字幕」，收音時請**關閉**同音校正），也一起評分

**開始前**：上方選單「執行階段 → 變更執行階段類型 → T4 GPU」→ 儲存。然後由上而下逐格按 ▶。

結果存在 `MEET/eval-runs/<時間>/`：`summary.md`（總表）、`report.json`（詳細數字）、各條件的辨識文字。
這是**封存測試**：只用來驗收，不根據它的錯誤調整詞彙或規則。


In [ ]:
# 第 1 格：設定
MEET_FOLDER = 'MEET'     # 在「我的雲端硬碟」中的資料夾名稱（找不到會自動搜尋）
ONLY = ''                # 只評估部分檔案，例如 '01'；空白＝全部
# full＝整個檔案一次辨識；vad10＝在停頓處切段、每段最長 10 秒；fixed10＝每 10 秒硬切（模擬即時字幕送 Workers AI）
CONDITIONS = 'full,vad10'
# 沿用先前結果（eval-runs 底下的資料夾名稱，可用逗號分隔多個）：只重新計分，不需重新辨識
REUSE = ''


In [ ]:
# 第 2 格：連接 Google Drive、找到 MEET、確認 GPU
import os, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive')
MEET = ROOT/MEET_FOLDER
if not MEET.is_dir():
    found = [Path(d)/n for d, dirs, _ in os.walk(ROOT) if Path(d).relative_to(ROOT).parts.__len__() < 3 for n in dirs if n == MEET_FOLDER]
    if not found:
        raise RuntimeError(f'找不到 {MEET_FOLDER} 資料夾，請確認名稱')
    MEET = found[0]
pairs = sorted(p.stem for p in MEET.glob('*.txt') if not p.stem.endswith('.chrome') and (MEET/(p.stem+'.mp3')).exists())
print('MEET 位置：', MEET, '｜測試組：', pairs, '｜Chrome 匯出：', sorted(p.name for p in MEET.glob('*.chrome.txt')))
import shutil
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() if shutil.which('nvidia-smi') else ''
print('GPU：', gpu or '沒有 GPU（請到「執行階段 → 變更執行階段類型」選 T4 GPU 後，從第 1 格重新執行；用 CPU 會非常慢）')


In [ ]:
# 第 3 格：取得固定版本的評估程式與套件（含 GPU 函式庫，約 2–3 分鐘）
import sys
COMMIT = '9d14789d9722bb12ac244ecd673881d02a078d81'
REPO = Path('/content/Meeting-Live-Captions')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/timo893131/Meeting-Live-Captions.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '-q', 'origin', COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faster-whisper==1.2.1', 'ctranslate2==4.8.3', 'av==16.0.1',
                'rapidfuzz==3.14.1', 'opencc-python-reimplemented==0.1.7', 'nodejs-wheel-binaries==22.20.0',
                # CTranslate2 needs CUDA 12 cuBLAS + cuDNN 9; newer Colab images do not ship libcublas.so.12.
                'nvidia-cublas-cu12==12.9.1.4', 'nvidia-cudnn-cu12==9.28.0.42'], check=True)
import importlib, importlib.util
importlib.invalidate_caches()
lib_dirs = [str(Path(location)/'lib') for module in ('nvidia.cublas', 'nvidia.cudnn')
            for location in importlib.util.find_spec(module).submodule_search_locations]
os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_dirs + [os.environ.get('LD_LIBRARY_PATH', '')])
print('CUDA 12 函式庫：', *lib_dirs, sep='\n  ')
import nodejs_wheel
os.environ['NODE_BINARY'] = str(Path(nodejs_wheel.__file__).parent/'bin'/'node')
print('程式版本', COMMIT[:12], '｜Node', subprocess.check_output([os.environ['NODE_BINARY'], '--version'], text=True).strip())


In [ ]:
# 第 4 格：下載固定版本的 Whisper large-v3-turbo 並核對 SHA-256（約 1.6GB）
import hashlib
from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download('deepdml/faster-whisper-large-v3-turbo-ct2', revision='4df90f75321148c3a29a9e2351b7ddf8f5b115a8',
                              allow_patterns=['config.json', 'model.bin', 'preprocessor_config.json', 'tokenizer.json', 'vocabulary.json'])
digest = hashlib.sha256()
with open(Path(MODEL_DIR)/'model.bin', 'rb') as stream:
    for block in iter(lambda: stream.read(1 << 24), b''):
        digest.update(block)
if digest.hexdigest() != 'e76620f83d5f5b69efd3d87e3dc180c1bd21df9fbebacfd4335e5e1efcc018da':
    raise RuntimeError('模型檔 SHA-256 不符，已停止')
print('模型已核對：', MODEL_DIR)


In [ ]:
# 第 5 格：執行評估（GPU 上約 5–15 分鐘）
from datetime import datetime, timezone
RUN = MEET/'eval-runs'/datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
command = [sys.executable, '-u', str(REPO/'scripts/evaluate_meet.py'), '--meet', str(MEET), '--output', str(RUN),
           '--model-dir', MODEL_DIR, '--model-label', 'whisper-large-v3-turbo (deepdml ct2 4df90f7)', '--conditions', CONDITIONS]
if ONLY:
    command += ['--only', ONLY]
for name in [n.strip() for n in REUSE.split(',') if n.strip()]:
    command += ['--reuse', str(MEET/'eval-runs'/name)]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait():
    raise RuntimeError('評估失敗，請看上面的錯誤訊息')
print('\n結果已存到：', RUN)


## 怎麼看結果

- **CER（字元錯誤率）**：越低越好。參考文字裡的經節出處（如「約四24」）和講員實際念法可能不同，`report.json` 另有去掉括號內容的 `no_parentheses` 版本。
- **專名正確**：參考文字中教會專名（書卷名、召會、經綸…）被正確辨識的次數；**人名地名正確**：聖經人名地名。
- **版本**：raw＝原始；glossary＝詞彙表；phonetic_core＝同音校正（書卷與教會專名）；phonetic＝再加上聖經人名地名。
- **只算講道表**：錄音結尾若有唱詩歌，另外列出不含詩歌的分數（唱歌被辨識成的文字不計）。
- **chrome_export**：MEET 裡有 NN.chrome.txt 時才會出現（做法見 docs/chrome-comparison.md）。
- **修好／改壞字數**：每一步校正相對上一步，多對了幾個字、多錯了幾個字。**改壞字數應該接近 0**。
- **whisper_live_vad_10s 等**：模擬即時字幕，每段獨立辨識、沒有前文，最接近實際送 Cloudflare Workers AI 的情況；整檔辨識（whisper_no_prompt）是上限。
- **錯誤分類表**：替換＝聽錯字；漏字＝參考文字有、辨識沒有；多字＝辨識多出來。**長段漏字（≥20 字）** 多半表示那段參考文字講者沒有念（或是音樂、雜音），不是辨識錯誤。
- **字幕延遲**：從一段開始說話到字幕出現的時間（段長＋處理時間），處理時間是 Colab T4 GPU 的數字，Workers AI 實際會不同（另加網路時間）。

請把 `summary.md` 的表格貼回對話，我會依結果建議下一步（例如是否加入 Workers AI 精修）。
